# Homework 3

In [1]:
from config import DIRECTORIES, DATA, TRAINING_CONFIG
from pathlib import Path
import random
import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(TRAINING_CONFIG.seed)

## (a) Tokenizer

We trained a BPE tokenizer with vocabulary size 3,000 on the first 200,000 training stories. BPE starts from a small inventory of characters or character-like pieces and repeatedly merges frequent adjacent pairs, so common fragments become single tokens while rare words still fall back to smaller subwords. This is preferable to pure character tokenization because the sequence length becomes much shorter, and it is preferable to full word tokenization because rare words and misspellings can still be represented without an exploding vocabulary.

For this notebook we used a metaspace BPE setup so spaces are preserved on decode, and we reserved the special tokens <pad>, <bos>, <eos>, and <unk>.

We also printed out 5 sentences from the first story, looked at the word count vs. token count, to see what tokens the tokenizer was picking up

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from typing import Iterable


def iter_stories(path: Path, limit: int | None = None) -> Iterable[str]:
    """Yields stories from the txt file one at a time"""
    buffer = ""
    count = 0
    with path.open("r", encoding="utf-8") as handle:
        while True:
            chunk = handle.read(8 * 1024 * 1024)
            if not chunk:
                break
            buffer += chunk
            pieces = buffer.split(DATA.story_delimiter)
            buffer = pieces.pop()
            for piece in pieces:
                story = piece.strip()
                if not story:
                    continue
                yield story
                count += 1
                if limit is not None and count >= limit:
                    return
        tail = buffer.strip()
        if tail and (limit is None or count < limit):
            yield tail


def build_tokenizer(train_path: Path, vocab_size: int, story_limit: int) -> Tokenizer:
    """Trains a tokenizer based on the inputted data"""
    # initialize
    tokenizer = Tokenizer(models.BPE(unk_token=DATA.special_tokens.unk))

    # define the metaspace and decoders
    tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(
        replacement="▁", prepend_scheme="always"
    )
    tokenizer.decoder = decoders.Metaspace(replacement="▁", prepend_scheme="always")

    # init the trainer
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        show_progress=False,
        special_tokens=DATA.special_tokens.all,
    )

    # train!
    tokenizer.train_from_iterator(
        iter_stories(train_path, limit=story_limit), trainer=trainer, length=story_limit
    )
    tokenizer.save(
        str(
            DIRECTORIES.tokenizers
            / f"tinystories_bpe_metaspace_{vocab_size}_{story_limit}.json"
        )
    )
    return tokenizer


tokenizer = build_tokenizer(
    DIRECTORIES.data / DATA.training_file,
    TRAINING_CONFIG.vocab_size,
    TRAINING_CONFIG.max_train_stories,
)


In [21]:
import re

for item in iter_stories(DIRECTORIES.data / DATA.training_file, 1):
    sentences = re.split(r"[.\n!?]+", item)
    for sentence in sentences[:5]:
        word_count = len(sentence.split(" "))
        encodings = tokenizer.encode(sentence)
        print(f"{sentence} has {word_count} words and {len(encodings)} tokens:")
        print(f"  {tuple(zip(encodings.tokens, encodings.ids))}")

Once upon a time there was a little boy named Ben has 11 words and 11 tokens:
  (('▁Once', 330), ('▁upon', 338), ('▁a', 158), ('▁time', 625), ('▁there', 303), ('▁was', 182), ('▁a', 158), ('▁little', 294), ('▁boy', 414), ('▁named', 304), ('▁Ben', 445))
 Ben loved to explore the world around him has 9 words and 8 tokens:
  (('▁Ben', 445), ('▁loved', 408), ('▁to', 165), ('▁explore', 1603), ('▁the', 162), ('▁world', 1604), ('▁around', 530), ('▁him', 379))
 He saw many amazing things, like beautiful vases that were on display in a store has 16 words and 20 tokens:
  (('▁He', 216), ('▁saw', 286), ('▁many', 575), ('▁amazing', 2090), ('▁thing', 477), ('s,', 430), ('▁like', 422), ('▁beautiful', 961), ('▁v', 593), ('as', 468), ('es', 296), ('▁that', 285), ('▁were', 308), ('▁on', 258), ('▁dis', 1222), ('pl', 744), ('ay', 179), ('▁in', 214), ('▁a', 158), ('▁store', 1299))
 One day, Ben was walking through the store when he came across a very special vase has 17 words and 17 tokens:
  (('▁One', 346

The above print / display statements illustrate the tokenization process on the first five sentences of the first story

## (b) Create the dataset

We concatenated tokenized stories into one long stream, appending <eos> after every story, and then chunked that stream into non-overlapping windows of length 128. For each chunk, the input is tokens t_0, ..., t_{L-1} and the target is the same chunk shifted by one position, which implements standard next token prediction.

The results were:
- **Train:** Train has 37781992 tokens and 295171 samples
- **Validation:** Train has 5184076 tokens and 40500 samples

In [ ]:
from torch.utils.data import Dataset


def count_tokens(
    tokenizer: Tokenizer, path: Path, story_limit: int | None = None
) -> int:
    total = 0
    for story in iter_stories(path, limit=story_limit):
        total += len(tokenizer.encode(story).ids) + 1  # +1 for the EOS Token!
    return total


def build_token_memmap(
    tokenizer: Tokenizer,
    path: Path,
    total_tokens: int,
    output_path: Path,
    story_limit: int | None = None,
) -> Path:
    """Returns the path of the built token memmap

    A memmap is basicallyl like a token stream, but optimized for quick retrieval"""
    if output_path.exists():
        return output_path

    # grab the EOS ID
    eos_id = tokenizer.token_to_id(DATA.special_tokens.eos)
    assert eos_id is not None

    # build the memmap, defining the shape up front
    token_array = np.memmap(
        output_path, dtype=np.uint16, mode="w+", shape=(total_tokens,)
    )

    # iterate through the dataset and build it as a stream of tokens
    offset = 0
    for story in iter_stories(path, limit=story_limit):
        ids = tokenizer.encode(story).ids

        # force a gap here
        next_offset = offset + len(ids) + 1
        token_array[offset : offset + len(ids)] = ids

        # put the EOS token in the gap
        token_array[offset + len(ids)] = eos_id
        offset = next_offset
    token_array.flush()

    return output_path


class TokenChunkDataset(Dataset):
    def __init__(self, token_path: Path, total_tokens: int, context_length: int):
        self.tokens = np.memmap(
            token_path, dtype=np.uint16, mode="r", shape=(total_tokens,)
        )
        self.context_length = context_length
        self.num_samples = (total_tokens - 1) // context_length

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns a tensor (input, target), like ((once, upon), (upon, a))"""
        start = idx * self.context_length
        end = start + self.context_length
        x = np.asarray(self.tokens[start:end], dtype=np.int64)
        y = np.asarray(self.tokens[start + 1 : end + 1], dtype=np.int64)
        return torch.from_numpy(x), torch.from_numpy(y)


train_token_count = count_tokens(
    tokenizer,
    DIRECTORIES.data / DATA.training_file,
    TRAINING_CONFIG.max_train_stories,
)
display(f"Token count for stories is {train_token_count}")
valid_token_count = count_tokens(
    tokenizer,
    DIRECTORIES.data / DATA.validation_file,
    TRAINING_CONFIG.max_train_stories,
)


train_token_memmap_path = (
    DIRECTORIES.tokenizers
    / f"train_tokens_metaspace_{TRAINING_CONFIG.max_train_stories}_v{TRAINING_CONFIG.vocab_size}.bin"
)
output_path = build_token_memmap(
    tokenizer,
    DIRECTORIES.data / DATA.training_file,
    train_token_count,
    train_token_memmap_path,
    TRAINING_CONFIG.max_train_stories,
)
display(f"Output train memmap at {output_path}")

valid_token_memmap_path = (
    DIRECTORIES.tokenizers
    / f"valid_tokens_metaspace_{TRAINING_CONFIG.max_train_stories}_v{TRAINING_CONFIG.vocab_size}.bin"
)
build_token_memmap(
    tokenizer,
    DIRECTORIES.data / DATA.validation_file,
    valid_token_count,
    valid_token_memmap_path,
    TRAINING_CONFIG.max_train_stories,
)

'Token count for stories is 37781992'

'Output train memmap at /Users/sanch/Obsidian/2025-spring/Notes/242B/Homeworks/hw3/artifacts/tokenizers/train_tokens_metaspace_200000_v3000.bin'

PosixPath('/Users/sanch/Obsidian/2025-spring/Notes/242B/Homeworks/hw3/artifacts/tokenizers/valid_tokens_metaspace_200000_v3000.bin')

In [24]:
train_dataset = TokenChunkDataset(
    train_token_memmap_path, train_token_count, TRAINING_CONFIG.context_window
)
valid_dataset = TokenChunkDataset(
    valid_token_memmap_path, valid_token_count, TRAINING_CONFIG.context_window
)

In [26]:
display(
    f"Train has {train_token_count} tokens and {int(train_token_count / TRAINING_CONFIG.context_window)} samples"
)
display(
    f"Validation has {valid_token_count} tokens and {int(valid_token_count / TRAINING_CONFIG.context_window)} samples"
)

'Train has 37781992 tokens and 295171 samples'

'Validation has 5184076 tokens and 40500 samples'